# Entrada y salida de datos con NumPy

Notebook guiado que acompaña a `12-numpy-io-archivos-avanzado.md`.

## Guardar / cargar un array binario `.npy`
`np.save` guarda un array en formato binario `.npy` (con metadatos de forma y dtype);
`np.load` lo recupera.

In [5]:
import numpy as np
from pathlib import Path
from tempfile import TemporaryDirectory
temporal = TemporaryDirectory(prefix='pia-numpy-')
carpeta = Path(temporal.name)

x = np.arange(12).reshape(3, 4)
np.save(carpeta / 'matriz.npy', x)

y = np.load(carpeta / 'matriz.npy')
print(y, y.shape, y.dtype)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]] (3, 4) int64


El formato `.npy` es **rápido** y conserva exactamente forma y tipo. Es la opción
recomendada para datos NumPy intermedios que no necesitan ser legibles por otros
programas.

## Varios arrays: `.npz` y `savez_compressed`
Para guardar varios arrays a la vez, `np.savez` y `np.savez_compressed` empaquetan en
un único archivo `.npz`. Al cargarlo, accedes por nombre.

In [6]:
a = np.array([1, 2, 3])
bl = np.ones((2, 2))
np.savez(carpeta / 'datos.npz', arr=a, mat=bl)

with np.load(carpeta / 'datos.npz') as datos:
    print('arr:', datos['arr'])
    print('mat:', datos['mat'])

arr: [1 2 3]
mat: [[1. 1.]
 [1. 1.]]


`np.savez_compressed` aplica compresión (archivo más pequeño, algo más lento). El
objeto devuelto por `np.load` para `.npz` es un *lazy*: lee de disco cuando accedes.

## Texto y CSV: `savetxt` / `loadtxt`
`np.savetxt` escribe arrays en texto legible; `np.loadtxt` los lee. Útil para CSV
sencillos y para datos que deban verse o editarse con un editor.

In [7]:
data = np.array([[1., 2., 3.],
                 [4., 5., 6.]])
np.savetxt(carpeta / 'data.csv', data, delimiter=',', fmt='%.3f')

leido = np.loadtxt(carpeta / 'data.csv', delimiter=',')
print(leido)

[[1. 2. 3.]
 [4. 5. 6.]]


Opciones típicas de `loadtxt`: `skiprows=n` para saltar filas de cabecera, `dtype=...`
para forzar tipo, y `delimiter=","` para CSV.

## Datos heterogéneos y faltantes: `genfromtxt`
Cuando el archivo mezcla texto y números, tiene cabeceras o celdas vacías, `genfromtxt`
es más flexible que `loadtxt`. Algunos fallos de importación se salvan dejando `nan`.

In [8]:
from io import StringIO

contenido = """\
id,edad,ciudad
1,32,Madrid
2,nd,Barcelona
3,45,Valencia
"""
buf = StringIO(contenido)
datos = np.genfromtxt(
    buf,
    delimiter=',',
    names=True,
    dtype=[('id', 'i8'), ('edad', 'f8'), ('ciudad', 'U20')],
    missing_values='nd',
    filling_values=np.nan,
)
print(datos)
print(datos['edad'])

[(1, 32., 'Madrid') (2, nan, 'Barcelona') (3, 45., 'Valencia')]
[32. nan 45.]


`names=True` toma la primera fila como nombres de campos; un `dtype` explícito por
columna permite que la edad sea `float` y acepte `nan`; `missing_values`/`filling_values`
convierten tokens (aquí `nd`) a `nan`. Si dejas `dtype=None` y hay un valor ausente en
una columna entera, `genfromtxt` falla al rellenar con `nan`; fija el `dtype` adecuado.

> `genfromtxt` es útil para datos sencillos. Para tablas complejas, missing-data
> avanzado o fechas, pandas (UD2) ofrece mucho más control.

## Binarios planos: `tofile` / `fromfile`
NumPy puede leer arrays en **archivos binarios planos** heredados con `fromfile` /
`tofile`, y mapear en memoria con `memmap`. El primero sirve para datos binarios que no
llevan metadatos; el segundo permite trabajar con conjuntos mayores que la RAM.

In [4]:
z = np.arange(9, dtype=np.float32)
z.tofile(carpeta / 'plano.bin')

leido = np.fromfile(carpeta / 'plano.bin', dtype=np.float32)
print(leido)

[0. 1. 2. 3. 4. 5. 6. 7. 8.]


Estos casos suelen aparecer en datos científicos o señales. Para datos tabulares,
pandas o parquet (UD2) son más cómodos.

## Limpieza de archivos temporales creados

Los ejemplos usan una carpeta temporal exclusiva para no sobrescribir archivos del directorio de trabajo. La siguiente celda elimina únicamente esa carpeta. Para repetir los ejemplos, ejecuta de nuevo desde la primera celda de código.

In [9]:
temporal.cleanup()
print('temporales borrados')

temporales borrados


## Reproducibilidad y ruta de trabajo

- Fija la carpeta o ruta al guardar; no uses rutas absolutas frágiles en prácticas.
- Al cargar, comprueba `shape` y `dtype` antes de operar.
- Para experimentos reproducibles, guarda también la semilla y los parámetros.

## Errores frecuentes

1. Usar `loadtxt` sobre un archivo con cabecera y olvidar `skiprows`.
2. Guardar con `np.save` y cargar con `np.loadtxt` (formatos incompatibles).
3. Olvidar el `dtype` al leer binarios planos con `fromfile`.
4. Ignorar que `np.savez(...)` pide arrays; si pasas un objeto no array puede fallar.
5. Suponer que `.npy` es legible como texto (no lo es).
6. No separar la conversión de tipos de la lectura en genfromtxt que mezcla texto.

## Ejercicios

1. Crea un array 4×5, guárdalo en `.npy` y recupéralo comprobando forma y dtype.
2. Guarda dos arrays en un `.npz` y recupéralos por nombre.
3. Escribe 10 filas de 3 columnas a un CSV y léelas con `loadtxt`.
4. Lee un CSV con cabecera y columna de texto con `genfromtxt(names=True)`.
5. Explica cuándo usarías `.npy` frente a CSV.